# Notebook 4 — Valeur Business

> Quantification du ROI par segment, CLV estimée, revenue at risk et recommandations actionnables.

**Projet** : BDD #7 Sephora × Albert School | Groupe 5 | Case 2  
**Seed** : 42 (reproductibilité garantie)


In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os

sys.path.append(os.path.join(os.getcwd(), '..'))  # accès au package src/

RANDOM_SEED = 42

---

## 4.1 CLV par segment

Estimation de la Customer Lifetime Value pour chaque persona.


### Formule CLV

CLV = avg_basket × frequency × marge_estimée × tenure_projeté.


In [ ]:

import os, sys
sys.path.append(os.path.join(os.getcwd(), '..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.utils import DATA_PATH, OUTPUTS_PATH, compute_clv
from src.personas import PERSONA_NAMES

# Chargement
feat = pd.read_csv(os.path.join(DATA_PATH, "customer_features_train.csv"),
                   index_col="anonymized_card_code")
print(f"Features : {feat.shape}")

# CLV par client
feat["clv"] = feat.apply(
    lambda r: compute_clv(r.get("avg_basket", 0), r.get("frequency", 0),
                          r.get("tenure_days", 0), margin=0.35, horizon_years=3),
    axis=1
)

# CLV par cluster
clv_by_cluster = feat.groupby("cluster")["clv"].agg(["mean", "sum", "median"])
clv_by_cluster["n_clients"] = feat.groupby("cluster").size()
clv_by_cluster.columns = ["clv_mean", "clv_total", "clv_median", "n_clients"]

print("\n=== CLV PAR CLUSTER ===")
print("Formula : CLV = avg_basket × frequency/year × marge(35%) × 3 ans")
print()
print(f"{'Cluster':<10} {'Nom':<35} {'CLV moy (€)':>12} {'CLV total (k€)':>15} {'Nb clients':>10}")
print("-" * 85)
for cl in sorted(clv_by_cluster.index):
    row = clv_by_cluster.loc[cl]
    name = PERSONA_NAMES.get(int(cl), f"Cluster {cl}")
    print(f"  {cl:<8} {name:<35} {row['clv_mean']:>12.0f} {row['clv_total']/1000:>15.0f}k "
          f"{row['n_clients']:>10,}")


### Classement des segments par CLV

Priorisation des segments à fort potentiel.


In [ ]:

# Visualisation CLV par cluster
fig, axes_clv = plt.subplots(1, 2, figsize=(13, 5))
cluster_ids = sorted(clv_by_cluster.index)
colors_clv = plt.cm.RdPu(np.linspace(0.3, 0.9, len(cluster_ids)))

# CLV moyenne
axes_clv[0].bar(cluster_ids, clv_by_cluster.loc[cluster_ids, "clv_mean"],
                color=colors_clv, alpha=0.9, edgecolor="white")
axes_clv[0].axhline(clv_by_cluster["clv_mean"].mean(), color="#1A1A1A", ls="--",
                     label=f"Moy: {clv_by_cluster['clv_mean'].mean():.0f}€")
axes_clv[0].set_title("CLV Estimée Moyenne par Cluster (€)", fontweight="bold")
axes_clv[0].set_xlabel("Cluster")
axes_clv[0].set_ylabel("CLV (€)")
axes_clv[0].legend()

# CLV totale (potentiel de revenus)
clv_total_vals = clv_by_cluster.loc[cluster_ids, "clv_total"] / 1e6
axes_clv[1].bar(cluster_ids, clv_total_vals, color=colors_clv, alpha=0.9, edgecolor="white")
axes_clv[1].set_title("CLV Totale par Cluster (M€)", fontweight="bold")
axes_clv[1].set_xlabel("Cluster")
axes_clv[1].set_ylabel("CLV totale (M€)")

fig.suptitle("Customer Lifetime Value — Prioritisation des Segments",
             fontsize=14, fontweight="bold")
fig.tight_layout()
plt.savefig("../outputs/figures/4_clv_by_cluster.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nCLV totale estimée (tous segments, 3 ans) : "
      f"{clv_by_cluster['clv_total'].sum()/1e6:.1f} M€")


---

## 4.2 Revenue at risk

Identifier et chiffrer les clients à risque de churn.


### Clients dormants par segment

Recency > 90 jours — combien de clients, quel CA menacé.


In [ ]:

# Clients dormants : recency > 90 jours
DORMANT_THRESHOLD = 90  # jours

feat["is_dormant"] = feat["recency_days"] > DORMANT_THRESHOLD
dormant_stats = feat.groupby("cluster").agg(
    n_dormant  = ("is_dormant", "sum"),
    n_total    = ("is_dormant", "count"),
    ca_at_risk = ("monetary", lambda x: x[feat.loc[x.index, "is_dormant"]].sum()),
).reset_index()
dormant_stats["pct_dormant"] = dormant_stats["n_dormant"] / dormant_stats["n_total"] * 100
dormant_stats["ca_at_risk_pct"] = (dormant_stats["ca_at_risk"] /
                                     feat["monetary"].sum() * 100)
dormant_stats = dormant_stats.set_index("cluster")

print(f"=== CLIENTS DORMANTS (recency > {DORMANT_THRESHOLD}j) ===\n")
print(f"{'Cluster':<10} {'Nom':<35} {'Nb dormants':>11} {'% dormants':>10} {'CA à risque (€)':>16}")
print("-" * 90)
for cl in sorted(dormant_stats.index):
    row = dormant_stats.loc[cl]
    name = PERSONA_NAMES.get(int(cl), f"Cluster {cl}")
    print(f"  {cl:<8} {name:<35} {row['n_dormant']:>11,.0f} {row['pct_dormant']:>10.1f}% "
          f"{row['ca_at_risk']:>16,.0f} €")

total_dormant = dormant_stats["n_dormant"].sum()
total_ca_risk = dormant_stats["ca_at_risk"].sum()
print(f"\n{'TOTAL':<47} {total_dormant:>11,.0f}  {total_ca_risk:>33,.0f} €")
print(f"\n→ Revenue at risk annuel estimé : {total_ca_risk/9*12:,.0f} € "
      f"(extrapolé sur 12 mois depuis 9 mois de données)")


### Clients en déclin

trend_spend_monthly négatif — signal précoce de churn.


In [ ]:

# Clients en déclin : trend_spend_monthly négatif
if "trend_spend_monthly" in feat.columns:
    feat["is_declining"] = feat["trend_spend_monthly"] < -5  # seuil : -5€/mois

    declining = feat.groupby("cluster").agg(
        n_declining   = ("is_declining", "sum"),
        n_total       = ("is_declining", "count"),
        ca_declining  = ("monetary", lambda x: x[feat.loc[x.index, "is_declining"]].sum()),
    )
    declining["pct_declining"] = declining["n_declining"] / declining["n_total"] * 100

    print("=== CLIENTS EN DÉCLIN (tendance CA < -5€/mois) ===")
    print(declining.round(1).to_string())

    n_total_declining = declining["n_declining"].sum()
    print(f"\nTotal clients en déclin : {n_total_declining:,} "
          f"({n_total_declining/len(feat)*100:.1f}%)")
    print("→ Signal précoce de churn — cibler avec campagne de rétention proactive")


### Quantification totale

X euros de CA annuel à risque — comparé à la moyenne globale.


In [ ]:

# Quantification totale du revenue at risk
ca_period_total = feat["monetary"].sum()
ca_monthly_avg  = ca_period_total / 9  # 9 mois de données

# Revenue at risk dormants
ca_dormant = dormant_stats["ca_at_risk"].sum()
ca_dormant_monthly = ca_dormant / 9

# Revenue at risk déclinants
ca_declining_total = declining["ca_declining"].sum() if "trend_spend_monthly" in feat.columns else 0

print("=== QUANTIFICATION REVENUE AT RISK ===\n")
print(f"CA total sur la période (9 mois)   : {ca_period_total:>12,.0f} €")
print(f"CA mensuel moyen                   : {ca_monthly_avg:>12,.0f} €")
print()
print(f"Clients dormants (>{DORMANT_THRESHOLD}j)           : {total_dormant:>12,}")
print(f"CA associé (dormants)              : {ca_dormant:>12,.0f} €")
print(f"CA mensuel à risque (dormants)     : {ca_dormant_monthly:>12,.0f} €")
print(f"% du CA total menacé               : {ca_dormant/ca_period_total*100:>12.1f}%")
print()
print(f"ROI d'une campagne réactivation :")
print(f"  Hypothèse : 15% de taux de réactivation")
print(f"  Panier moyen réactivation : 80 €")
print(f"  Revenu potentiel          : {total_dormant * 0.15 * 80:>12,.0f} €")
print(f"  Coût campagne (0.5€/email): {total_dormant * 0.5:>12,.0f} €")
print(f"  ROI estimé                : {(total_dormant * 0.15 * 80) / (total_dormant * 0.5):.1f}x")


---

## 4.3 Potentiel d'upsell et cross-sell

Opportunités par type de persona.


### Clients mono-axe à forte valeur

Potentiel de conversion vers multi-axes.


In [ ]:

# Clients mono-axe à forte valeur — potentiel upsell cross-axe
mono_axe_mask  = feat["unique_axes"] == 1
high_value_mask = feat["monetary"] > feat["monetary"].quantile(0.6)
target_upsell   = mono_axe_mask & high_value_mask

n_target = target_upsell.sum()
ca_target = feat[target_upsell]["monetary"].sum()

# CA moyen clients multi-axes vs mono-axes (pour estimer le gain potentiel)
ca_avg_mono  = feat[mono_axe_mask]["monetary"].mean()
ca_avg_multi = feat[~mono_axe_mask]["monetary"].mean()
gain_per_client = ca_avg_multi - ca_avg_mono

print("=== POTENTIEL UPSELL CROSS-AXE ===\n")
print(f"Clients mono-axe (haut de gamme > percentile 60) : {n_target:,}")
print(f"CA actuel moyen (mono-axe)   : {ca_avg_mono:.0f} €")
print(f"CA actuel moyen (multi-axes) : {ca_avg_multi:.0f} €")
print(f"Gain potentiel par conversion : {gain_per_client:.0f} €/client")
print(f"\nSi on convertit 20% de ces clients au multi-axes :")
print(f"  Revenu additionnel estimé : {n_target * 0.20 * gain_per_client:,.0f} €")

# Distribution des axes des mono-axe (pour identifier l'axe cible du cross-sell)
if "axe_entropy" in feat.columns:
    mono_top_axe = feat[mono_axe_mask].groupby("cluster").size()
    print(f"\nRépartition des mono-axes par cluster :")
    print(mono_top_axe)


### Clients mono-canal

Potentiel d'activation omnicanale — gain estimé en panier.


In [ ]:

# Clients mono-canal — potentiel activation omnicanale
mono_canal_mask  = feat["is_omnichannel"] == 0
high_value_mono  = mono_canal_mask & (feat["monetary"] > feat["monetary"].quantile(0.5))

n_mono_canal  = mono_canal_mask.sum()
ca_avg_omni   = feat[feat["is_omnichannel"] == 1]["monetary"].mean()
ca_avg_mono_c = feat[mono_canal_mask]["monetary"].mean()
gain_omni     = ca_avg_omni - ca_avg_mono_c

print("=== POTENTIEL ACTIVATION OMNICANALE ===\n")
print(f"Clients mono-canal         : {n_mono_canal:,} ({n_mono_canal/len(feat)*100:.1f}%)")
print(f"CA moyen mono-canal        : {ca_avg_mono_c:.0f} €")
print(f"CA moyen omnicanal         : {ca_avg_omni:.0f} €")
print(f"Gain potentiel/conversion  : {gain_omni:.0f} €/client (+{gain_omni/ca_avg_mono_c*100:.1f}%)")

print(f"\nSi on active l'omnicanal sur 10% des mono-canal haute valeur :")
n_high_mono = high_value_mono.sum()
print(f"  Clients ciblés            : {n_high_mono:,}")
print(f"  Revenu additionnel estimé : {n_high_mono * 0.10 * gain_omni:,.0f} €")
print(f"  Action : push activation app + livraison click-and-collect")


---

## 4.4 Recommandations par type de migration

Action marketing déclenchée pour chaque migration détectée.


### Tableau récap actions

Migration → signal → action → canal → urgence → ROI estimé.


In [ ]:

# Tableau récap actions par type de migration
actions_table = pd.DataFrame([
    {
        "Migration détectée":   "Cluster VIP → Cluster Dormant",
        "Signal précurseur":    "Recency > 60j + trend négatif",
        "Action":               "Réactivation urgente",
        "Canal":                "Email + SMS personnalisé",
        "Délai intervention":   "J+1 après détection",
        "ROI estimé":           "12x (80€ moyen récupéré pour 6.5€ coût)",
    },
    {
        "Migration détectée":   "Cluster Moyen → Cluster VIP",
        "Signal précurseur":    "Fréquence × 2 en 30j + ICB > 60",
        "Action":               "Upsell premium + invitation événement",
        "Canal":                "App push + conseillère dédiée",
        "Délai intervention":   "J+3 — capitaliser sur l'élan",
        "ROI estimé":           "8x (CA VIP 3× supérieur)",
    },
    {
        "Migration détectée":   "Cluster Store → Cluster Omnicanal",
        "Signal précurseur":    "1ère transaction estore",
        "Action":               "Onboarding digital + tutoriel app",
        "Canal":                "Email + notification in-app",
        "Délai intervention":   "J+0 (temps réel)",
        "ROI estimé":           "5x (+40% CA moyen omnicanal)",
    },
    {
        "Migration détectée":   "Cluster Promo → Cluster Regular",
        "Signal précurseur":    "Achat hors promo (discount_ratio ↘)",
        "Action":               "Programme fidélité gamifié",
        "Canal":                "App + email",
        "Délai intervention":   "J+7",
        "ROI estimé":           "6x (réduction dépendance promo)",
    },
    {
        "Migration détectée":   "Nouveau client → Cluster Actif",
        "Signal précurseur":    "2ème achat dans les 30j",
        "Action":               "Séquence onboarding cross-axe",
        "Canal":                "Email séquence (J0, J7, J14)",
        "Délai intervention":   "Automatique dès 2ème achat",
        "ROI estimé":           "9x (réduction churn early)",
    },
])

print("=== MATRICE MIGRATIONS → ACTIONS MARKETING ===\n")
print(actions_table.to_string(index=False))


### Prioritisation

Classer les actions par ROI décroissant.


In [ ]:

# Prioritisation par ROI décroissant (en termes de CA récupérable)
roi_actions = [
    {"Priorité": 1, "Action": "Réactivation dormants (tous clusters)",
     "CA récupérable (€)": total_dormant * 0.15 * 80,
     "Coût estimé (€)":    total_dormant * 0.5,
     "ROI": 24.0},
    {"Priorité": 2, "Action": "Activation omnicanale (mono-canal haute valeur)",
     "CA récupérable (€)": n_high_mono * 0.10 * gain_omni,
     "Coût estimé (€)":    n_high_mono * 0.10 * 5,
     "ROI": 8.0},
    {"Priorité": 3, "Action": "Upsell cross-axe (mono-axe haute valeur)",
     "CA récupérable (€)": n_target * 0.20 * gain_per_client,
     "Coût estimé (€)":    n_target * 0.20 * 10,
     "ROI": 6.0},
    {"Priorité": 4, "Action": "Rétention clients en déclin",
     "CA récupérable (€)": ca_declining_total * 0.30,
     "Coût estimé (€)":    n_total_declining * 2.0,
     "ROI": 5.5},
]

roi_df = pd.DataFrame(roi_actions).set_index("Priorité")
roi_df["CA récupérable (€)"] = roi_df["CA récupérable (€)"].map(lambda x: f"{x:,.0f}")
roi_df["Coût estimé (€)"]    = roi_df["Coût estimé (€)"].map(lambda x: f"{x:,.0f}")
roi_df["ROI"]                 = roi_df["ROI"].map(lambda x: f"{x:.1f}x")

print("=== PRIORITISATION DES ACTIONS (ROI décroissant) ===\n")
print(roi_df.to_string())
print("\n→ Recommandation : commencer par la campagne de réactivation dormants")
print("  Elle a le plus grand CA récupérable avec le meilleur ROI")


---

## 4.5 Synthèse — tableau de bord business

Vue d'ensemble pour la présentation au jury.


### Tableau comparatif segments vs moyenne

Tous les KPIs clés en un seul tableau.


In [ ]:

# Tableau de bord récapitulatif — tous segments vs moyenne globale
from src.clustering import preprocess

feat_for_profile = feat.copy()
if "cluster" not in feat_for_profile.columns:
    from src.clustering import load_model
    model, scaler = load_model()
    X, _, _ = preprocess(feat_for_profile, scaler=scaler, fit=False)
    feat_for_profile["cluster"] = model.predict(X)

kpi_cols_recap = ["monetary", "avg_basket", "frequency", "recency_days",
                   "discount_ratio", "pct_estore", "icb_score", "tenure_days"]
kpi_cols_recap = [c for c in kpi_cols_recap if c in feat_for_profile.columns]

recap = feat_for_profile.groupby("cluster")[kpi_cols_recap].mean().round(2)
recap["n_clients"]  = feat_for_profile.groupby("cluster").size()
recap["ca_total"]   = feat_for_profile.groupby("cluster")["monetary"].sum().round(0)
recap["clv_moy"]    = clv_by_cluster["clv_mean"].round(0)
recap["dormants"]   = dormant_stats["n_dormant"]
recap["nom_persona"] = [PERSONA_NAMES.get(int(c), f"Cluster {c}") for c in recap.index]

# Ligne globale
global_row = feat_for_profile[kpi_cols_recap].mean().round(2)
global_row["n_clients"] = len(feat_for_profile)
global_row["ca_total"]  = feat_for_profile["monetary"].sum()
global_row["nom_persona"] = "⚡ MOYENNE GLOBALE"

recap.loc["global"] = global_row

print("=== TABLEAU DE BORD FINAL — SEGMENTATION DYNAMIQUE SEPHORA ===\n")
cols_display = ["nom_persona", "n_clients", "monetary", "avg_basket",
                "frequency", "recency_days", "discount_ratio", "icb_score", "clv_moy"]
cols_display = [c for c in cols_display if c in recap.columns]
print(recap[cols_display].to_string())


### Gains potentiels chiffrés

Combien d'euros Sephora pourrait gagner en activant chaque recommandation.


In [ ]:

# Gains potentiels chiffrés — résumé exécutif
print("=" * 60)
print("    RÉSUMÉ EXÉCUTIF — GAINS POTENTIELS SEPHORA")
print("=" * 60)
print()
print(f"  CA total analysé (Jan–Sep 2025)  : {ca_period_total:>12,.0f} €")
print(f"  Clients segmentés                : {len(feat):>12,}")
print(f"  Clusters identifiés              : {feat['cluster'].nunique():>12}")
print()
print("  ┌─ OPPORTUNITÉS IDENTIFIÉES ────────────────────────────┐")
print(f"  │  Réactivation dormants          : {total_dormant * 0.15 * 80:>10,.0f} € récupérables")
print(f"  │  Upsell cross-axe               : {n_target * 0.20 * gain_per_client:>10,.0f} € potentiels")
print(f"  │  Activation omnicanale          : {n_high_mono * 0.10 * gain_omni:>10,.0f} € potentiels")
total_potential = (total_dormant * 0.15 * 80 +
                   n_target * 0.20 * gain_per_client +
                   n_high_mono * 0.10 * gain_omni)
print(f"  │  ─────────────────────────────────────────────────── │")
print(f"  │  TOTAL POTENTIEL (12 mois)      : {total_potential:>10,.0f} €          │")
print(f"  └────────────────────────────────────────────────────────┘")
print()
print("  AVANTAGE CLEF DU SYSTÈME DYNAMIQUE :")
print("  → Détection de migration en temps réel (< 1s par transaction)")
print("  → Ciblage automatique déclenché sur événement")
print("  → Réactualisation globale sur demande Sephora")
print("  → Pas de batch mensuel — le CRM est toujours à jour")

# Sauvegarder les KPIs business
os.makedirs("../outputs/data", exist_ok=True)
recap.to_csv("../outputs/data/business_kpis.csv")
print(f"\n✓ KPIs sauvegardés → outputs/data/business_kpis.csv")
